In [1]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://54.174.250.242:5000/")

In [3]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='s3://yt-mlflow/855456597129840563', creation_time=1743560523248, experiment_id='855456597129840563', last_update_time=1743560523248, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}>

In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna


In [5]:
df = pd.read_csv('reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [6]:
# Step 1: (Optional) Remapping - skipped since not strictly needed for KNN

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for KNN
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 3, 30)  # Tuning the number of neighbors
    p = trial.suggest_categorical('p', [1, 2])  # Tuning the distance metric (1 for Manhattan, 2 for Euclidean)

    # KNeighborsClassifier setup
    model = KNeighborsClassifier(n_neighbors=n_neighbors, p=p)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for KNN, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_knn, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = KNeighborsClassifier(n_neighbors=best_params['n_neighbors'], p=best_params['p'])

    # Log the best model with MLflow, passing the algo_name as "KNN"
    log_mlflow("KNN", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for KNN
run_optuna_experiment()


[I 2025-04-02 07:59:34,580] A new study created in memory with name: no-name-5aa4b88f-4701-40e9-8c05-813e787533e5
[I 2025-04-02 07:59:42,105] Trial 0 finished with value: 0.39315155358275206 and parameters: {'n_neighbors': 11, 'p': 1}. Best is trial 0 with value: 0.39315155358275206.
[I 2025-04-02 08:00:12,511] Trial 1 finished with value: 0.572711900232509 and parameters: {'n_neighbors': 9, 'p': 2}. Best is trial 1 with value: 0.572711900232509.
[I 2025-04-02 08:00:45,543] Trial 2 finished with value: 0.5397378989642782 and parameters: {'n_neighbors': 21, 'p': 2}. Best is trial 1 with value: 0.572711900232509.
[I 2025-04-02 08:01:16,132] Trial 3 finished with value: 0.5600295920524202 and parameters: {'n_neighbors': 13, 'p': 2}. Best is trial 1 with value: 0.572711900232509.
[I 2025-04-02 08:01:22,298] Trial 4 finished with value: 0.3913548932572395 and parameters: {'n_neighbors': 12, 'p': 1}. Best is trial 1 with value: 0.572711900232509.
[I 2025-04-02 08:01:55,376] Trial 5 finished 

MlflowException: API request to http://54.174.250.242:5000/api/2.0/mlflow/runs/create failed with exception HTTPConnectionPool(host='54.174.250.242', port=5000): Max retries exceeded with url: /api/2.0/mlflow/runs/create (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001F863542E50>: Failed to establish a new connection: [WinError 10065] A socket operation was attempted to an unreachable host'))